# H1 Xu first-level decomposition — revised grouped bars

Scientific authority: the user-supplied **`h1_xu_first_level_standalone.py`**, confirmed by the user as the updated analysis. Visual authority: **`East_River_Plotting_Requirements_JHM_AMS_v5.md`**. This is a new notebook; the original and upstream inputs are unchanged.

## Scientific definitions and interpretation

I = iSnobal (reference); N = Noah-MP. For each model, the initial state is **October 1 end-of-day SWE**, the endpoint is **March 15 SWE**, and snowfall is summed **October 2–March 15 inclusive**. October 1 snowfall is already represented in the initial state.

\[
G_j=\mathrm{SWE}_{\mathrm{Mar15},j}-\mathrm{SWE}_{\mathrm{Oct01},j},\qquad
S_j=\sum_{\mathrm{Oct02}}^{\mathrm{Mar15}}\mathrm{snowfall}_j,\qquad
M_j=1-G_j/S_j.
\]
\[
\epsilon_T=G_N/G_I-1,\quad \epsilon_A=S_N/S_I-1,\quad
\epsilon_M=(M_I-M_N)/(1-M_I),\quad
1+\epsilon_T=(1+\epsilon_A)(1+\epsilon_M).
\]

Bars show **100 εA** and **100 εM**; the connected markers show **100 εT**. These are dimensionless relative differences expressed as **percent**, not absolute contribution shares. The terms are multiplicative, so the bars must not be stacked or added as a total. With the positive reference gains in the supplied CSV, negative εT means Noah-MP has less corrected SWE gain than iSnobal; negative εM means a larger inferred ablation fraction in Noah-MP. M is **not an explicit model melt flux**. No clipping to [0, 1] is performed.

The older v0.8.2 Nov 1–Mar 15 additive initial-state/snowfall/retention H1 is a different analysis and is not substituted here. Plotting preserves the CSV's years, order after chronological sorting, signed values, upstream spatial support/weighting, and the original ×100 conversion. It does not recalculate spatial means, masks, or daily aggregation.

## Review and lineage

The supplied nine-year CSV (WY 2018–2026) was independently reproduced by running the unmodified supplied Xu script on local copies of `isnobal_watershed_daily.csv` and `noahmp_watershed_daily.csv` from workflow output1. Maximum numerical difference across numeric columns: 5.69 × 10⁻¹⁴; maximum direct multiplicative residual: 8.33 × 10⁻¹⁷. This checks the supplied files, not the current remote filesystem or raw model processing. Exact reviewed paths and SHA-256 hashes are in the companion review report.

The source helper `complete_sum` says “finite” but implements `notna()`, which would count infinity as present; its incomplete-coverage warning also overstates that all Xu metrics become NaN (the SWE-gain total can remain finite). Neither issue affects this supplied CSV: all required values are finite, coverage counts agree, and both M flags are false. The source's supplied water-year labels are trusted rather than checked against dates. The available daily tables were checked separately for this issue.

For future input, the validation below stops on empty/duplicate/noninteger years, undefined values, incomplete snowfall coverage, inconsistent dates, or failed identities. It never silently removes years or replaces values. Out-of-range M values are warned about and retained.


In [ ]:
from __future__ import annotations

from pathlib import Path
import hashlib
import json
import warnings

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
import numpy as np
import pandas as pd


## Explicit paths

The production output directory below is the one supplied by the user. The metrics and source script retain their original workflow-root placement; change these two paths if you store the updated Xu files elsewhere. No search for alternate CSVs or replacement of the input dataset is performed.

For a local Windows run, set `METRICS_PATH = Path.home() / "Downloads/h1_xu_first_level_metrics.csv"`, set `XU_SOURCE_PATH` to the downloaded script, choose a local `FIGURE_OUTPUT_DIR`, and set `NIMBUS_FONT_PATH` to a regular Nimbus Sans font file if the font is not installed. The font is registered only for the current process; no fallback is permitted.


In [ ]:
WORKFLOW_ROOT = Path(
    "/bsuhome/zihaozhang/scratch/iSnobal/East_River_Workflow_Code_v0.8.2"
)
WORKFLOW_OUTPUT_DIR = WORKFLOW_ROOT / "output" / "output1"
METRICS_PATH = WORKFLOW_ROOT / "h1_xu_first_level_metrics.csv"
XU_SOURCE_PATH = WORKFLOW_ROOT / "h1_xu_first_level_standalone.py"
FIGURE_OUTPUT_DIR = WORKFLOW_OUTPUT_DIR / "figures" / "scientific"
FIGURE_STEM = "h1_xu_first_level_grouped_bar_revised"

# Optional explicit font file; otherwise Nimbus Sans must already be available.
NIMBUS_FONT_PATH = None  # e.g. Path("/path/to/NimbusSans-Regular.ttf")

# These encode processes, NOT separate model series. Preserve original identity.
PROCESS_COLORS = {"accumulation": "#7A6FAC", "ablation": "#D98C3F"}
TOTAL_COLOR = "black"  # derived comparison, not an observation series
# Reserved for actual source/model series; not mapped to the two process bars.
MODEL_COLORS = {"observations": "black", "ASO": "black",
                "iSnobal": "#1F77B4", "Noah-MP": "#D62728"}
FRACTION_ATOL = 1e-10  # original closure tolerance; rtol=0 throughout
MM_ATOL = 1e-8         # roundoff tolerance for independently audited mm identities


In [ ]:
def configure_publication_style(font_path=None):
    """Require and resolve the real normal-weight Nimbus Sans face."""
    if font_path is not None:
        font_path = Path(font_path).expanduser()
        if not font_path.is_file():
            raise FileNotFoundError(f"Nimbus Sans font file not found: {font_path}")
        fm.fontManager.addfont(str(font_path))
    name = "Nimbus Sans"
    if name not in {f.name for f in fm.fontManager.ttflist}:
        raise RuntimeError(
            "Nimbus Sans is required. Install it or set NIMBUS_FONT_PATH to its "
            "regular font file, then rerun this cell. No fallback is allowed."
        )
    resolved = fm.findfont(fm.FontProperties(family=[name], style="normal",
                                           weight="normal"), fallback_to_default=False)
    if fm.FontProperties(fname=resolved).get_name() != name:
        raise RuntimeError(f"Unexpected font resolved: {resolved}")
    mpl.rcParams.update({
        "font.family": name, "font.size": 18, "font.weight": "normal",
        "axes.labelsize": 18, "axes.labelweight": "normal",
        "axes.titlesize": 20, "axes.titleweight": "normal",
        "xtick.labelsize": 18, "ytick.labelsize": 18, "legend.fontsize": 18,
        "text.usetex": False, "axes.formatter.use_mathtext": False,
        "pdf.fonttype": 42, "ps.fonttype": 42, "pdf.use14corefonts": False,
        "figure.facecolor": "white", "savefig.facecolor": "white",
        "savefig.transparent": False, "figure.dpi": 120, "savefig.dpi": 600,
        "axes.grid": False, "xtick.direction": "out", "ytick.direction": "out",
    })
    return Path(resolved)


def validate_xu_metrics(raw):
    """Audit the supplied definitions without replacing the values plotted."""
    required = {
        "water_year", "H1_start_state_date", "H1_flux_start_date", "H1_end_state_date",
        "epsilon_total_corrected", "epsilon_accumulation", "epsilon_ablation",
        "epsilon_total_reconstructed", "initial_state_closure_error_mm",
        "xu_multiplicative_closure_error", "Delta_SWE_Oct01_N_minus_I_mm",
        "Delta_SWE_Mar15_N_minus_I_mm", "Delta_corrected_SWE_gain_N_minus_I_mm",
    }
    for s in ("I", "N"):
        required.update({f"SWE_Oct01_{s}_mm", f"SWE_Mar15_{s}_mm",
                         f"SWE_gain_Oct01_Mar15_{s}_mm", f"Snowfall_Oct02_Mar15_{s}_mm",
                         f"M_ablation_{s}", f"snowfall_complete_{s}",
                         f"snowfall_expected_days_{s}", f"snowfall_valid_days_{s}",
                         f"M_{s}_outside_0_1"})
    missing = required - set(raw.columns)
    if missing:
        raise ValueError(f"Missing Xu audit columns: {sorted(missing)}")
    if raw.empty:
        raise ValueError("The Xu table is empty.")
    df = raw.copy()
    date_cols = {"H1_start_state_date", "H1_flux_start_date", "H1_end_state_date"}
    bool_cols = {f"{prefix}_{s}" for s in ("I", "N") for prefix in ("snowfall_complete",)}
    bool_cols |= {f"M_{s}_outside_0_1" for s in ("I", "N")}
    for col in sorted(required - date_cols - bool_cols):
        df[col] = pd.to_numeric(df[col], errors="raise")
        if not np.isfinite(df[col].to_numpy(dtype=float)).all():
            raise ValueError(f"Undefined/nonfinite {col}; resolve upstream before export. No rows dropped.")
    years = df["water_year"].to_numpy(dtype=float)
    if np.any(years != np.floor(years)) or df["water_year"].duplicated().any():
        raise ValueError("Each water year must be a unique integer.")
    df["water_year"] = df["water_year"].astype(int)
    df = df.sort_values("water_year", kind="stable").reset_index(drop=True)
    for col in bool_cols:
        normalized = df[col].astype(str).str.lower()
        if not normalized.isin(["true", "false"]).all():
            raise ValueError(f"Invalid Boolean values in {col}.")
        df[col] = normalized.eq("true")

    expected_start = pd.to_datetime((df.water_year - 1).astype(str) + "-10-01")
    expected_flux = expected_start + pd.Timedelta(days=1)
    expected_end = pd.to_datetime(df.water_year.astype(str) + "-03-15")
    for col, expected in zip(sorted(date_cols), [expected_end, expected_flux, expected_start]):
        actual = pd.to_datetime(df[col], errors="raise")
        if not actual.eq(expected).all():
            raise ValueError(f"{col} does not match the supplied Xu window.")
    expected_days = (expected_end - expected_flux).dt.days + 1
    for s in ("I", "N"):
        if (not df[f"snowfall_complete_{s}"].all()
            or not df[f"snowfall_expected_days_{s}"].eq(expected_days).all()
            or not df[f"snowfall_valid_days_{s}"].eq(expected_days).all()):
            raise ValueError(f"Incomplete/inconsistent snowfall coverage for {s}; no rows dropped.")

    residuals = {}
    def check(name, actual, expected, atol=FRACTION_ATOL):
        actual = np.asarray(actual, dtype=float)
        expected = np.asarray(expected, dtype=float)
        if not (np.isfinite(actual).all() and np.isfinite(expected).all()):
            raise ValueError(f"Undefined audit identity: {name}")
        residuals[name] = float(np.max(np.abs(actual - expected)))
        if not np.allclose(actual, expected, rtol=0, atol=atol):
            raise ValueError(f"{name} failed: max absolute residual {residuals[name]:.3e}")

    gains, snow, ablation = {}, {}, {}
    for s in ("I", "N"):
        gains[s] = df[f"SWE_Mar15_{s}_mm"] - df[f"SWE_Oct01_{s}_mm"]
        snow[s] = df[f"Snowfall_Oct02_Mar15_{s}_mm"]
        if snow[s].eq(0).any():
            raise ValueError(f"Zero snowfall denominator for {s}; metric undefined.")
        check(f"SWE gain {s}", df[f"SWE_gain_Oct01_Mar15_{s}_mm"], gains[s], MM_ATOL)
        ablation[s] = 1 - gains[s] / snow[s]
        check(f"M {s}", df[f"M_ablation_{s}"], ablation[s])
        outside = (df[f"M_ablation_{s}"] < 0) | (df[f"M_ablation_{s}"] > 1)
        if not outside.eq(df[f"M_{s}_outside_0_1"]).all():
            raise ValueError(f"Inconsistent M flag for {s}.")
        if outside.any():
            warnings.warn(f"M_{s} outside [0,1] in WY {df.loc[outside, 'water_year'].tolist()}; retained without clipping.")
    if gains["I"].eq(0).any() or (1 - ablation["I"]).eq(0).any():
        raise ValueError("Zero reference denominator; Xu relative difference is undefined.")
    check("epsilon total", df.epsilon_total_corrected, gains["N"] / gains["I"] - 1)
    check("epsilon accumulation", df.epsilon_accumulation, snow["N"] / snow["I"] - 1)
    check("epsilon ablation", df.epsilon_ablation,
          (ablation["I"] - ablation["N"]) / (1 - ablation["I"]))
    reconstructed = (1 + df.epsilon_accumulation) * (1 + df.epsilon_ablation) - 1
    check("direct multiplicative closure", df.epsilon_total_corrected, reconstructed)
    check("stored reconstructed total", df.epsilon_total_reconstructed, reconstructed)
    check("stored Xu residual", df.xu_multiplicative_closure_error,
          df.epsilon_total_corrected - reconstructed)
    delta_start = df.SWE_Oct01_N_mm - df.SWE_Oct01_I_mm
    delta_end = df.SWE_Mar15_N_mm - df.SWE_Mar15_I_mm
    delta_gain = gains["N"] - gains["I"]
    for col, expected in [("Delta_SWE_Oct01_N_minus_I_mm", delta_start),
                          ("Delta_SWE_Mar15_N_minus_I_mm", delta_end),
                          ("Delta_corrected_SWE_gain_N_minus_I_mm", delta_gain)]:
        check(col, df[col], expected, MM_ATOL)
    check("initial-state identity", delta_end, delta_start + delta_gain, MM_ATOL)
    check("stored initial-state residual", df.initial_state_closure_error_mm,
          delta_end - delta_start - delta_gain, MM_ATOL)
    # Plot supplied values, not recalculated audit values. Fractions -> percent.
    df["epsilon_total_pct"] = 100.0 * df["epsilon_total_corrected"]
    df["epsilon_accum_pct"] = 100.0 * df["epsilon_accumulation"]
    df["epsilon_ablation_pct"] = 100.0 * df["epsilon_ablation"]
    return df, residuals


In [ ]:
def plot_xu(df):
    fig, ax = plt.subplots(figsize=(14, 7.5))
    ax.set_facecolor("0.94")
    ax.set_axisbelow(True)
    ax.yaxis.grid(True, color="0.5", linestyle="--", linewidth=0.8, alpha=0.45)
    ax.xaxis.grid(False)
    x = np.arange(len(df), dtype=float)
    width = 0.34
    for offset, column, process, hatch in [
        (-width / 2, "epsilon_accum_pct", "accumulation", ""),
        (width / 2, "epsilon_ablation_pct", "ablation", "//"),
    ]:
        ax.bar(x + offset, df[column], width=width, color=PROCESS_COLORS[process],
               edgecolor="black", linewidth=1.0, hatch=hatch, zorder=3)
    ax.plot(x, df.epsilon_total_pct, color=TOTAL_COLOR, lw=2.0, marker="o",
            markersize=8, markerfacecolor="white", markeredgecolor=TOTAL_COLOR,
            markeredgewidth=1.5, zorder=4)
    ax.axhline(0, color="0.35", lw=1.0, zorder=2)
    ax.set_xticks(x, [str(wy) for wy in df.water_year])
    ax.set_xlabel("Water year")
    ax.set_ylabel("Relative difference (%)")
    # Preserve original 5-percentage-point tick spacing and dynamic signed limits.
    values = np.concatenate([df.epsilon_total_pct, df.epsilon_accum_pct,
                             df.epsilon_ablation_pct, [0.0]])
    data_min, data_max = float(values.min()), float(values.max())
    pad = 0.10 * max(data_max - data_min, 5.0)
    ax.set_ylim(np.floor((min(0, data_min) - pad) / 5) * 5,
                np.ceil((max(0, data_max) + pad) / 5) * 5)
    ax.yaxis.set_major_locator(MultipleLocator(5.0))
    ax.spines[["top", "right"]].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_linewidth(1.0)
    ax.tick_params(axis="both", which="major", width=1.0, length=5)
    handles = [
        Patch(facecolor=PROCESS_COLORS["accumulation"], edgecolor="black",
              linewidth=1.0, label="Accumulation term"),
        Patch(facecolor=PROCESS_COLORS["ablation"], edgecolor="black",
              linewidth=1.0, hatch="//", label="Ablation term"),
        Line2D([], [], color=TOTAL_COLOR, lw=2.0, marker="o", markersize=8,
               markerfacecolor="white", markeredgecolor=TOTAL_COLOR,
               markeredgewidth=1.5, label="Corrected SWE-gain difference"),
    ]
    # Lower left avoids near-zero bars and total markers in the reviewed data.
    # Keep the v5 boxed treatment; reassess placement for materially different data.
    ax.legend(handles=handles, loc="lower left", ncol=1, frameon=True,
              facecolor="white", edgecolor="0.4", framealpha=0.9,
              fancybox=False, borderpad=0.5, handlelength=2.2)
    # Single panel: no automatic main title or water-year panel title.
    fig.subplots_adjust(top=0.97, bottom=0.15, left=0.11, right=0.98)
    return fig, ax


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def export_xu(fig, out_dir, stem, df, residuals, font_path):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    png_path, pdf_path = out_dir / f"{stem}.png", out_dir / f"{stem}.pdf"
    fig.savefig(png_path, dpi=600, bbox_inches="tight", pad_inches=0.05)
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight", pad_inches=0.05)
    provenance = {
        "metrics_path": str(METRICS_PATH.resolve()),
        "metrics_sha256": sha256_file(METRICS_PATH),
        "xu_source_path": str(XU_SOURCE_PATH),
        "xu_source_sha256": sha256_file(XU_SOURCE_PATH) if XU_SOURCE_PATH.is_file() else None,
        "workflow_output_dir": str(WORKFLOW_OUTPUT_DIR),
        "font_path": str(font_path), "font_sha256": sha256_file(font_path),
        "water_years": df.water_year.tolist(), "row_count": len(df),
        "audit_max_absolute_residuals": residuals,
        "audit_scope": "CSV identities and metadata; no raw-model reprocessing",
        "scientific_authority": "user-supplied updated h1_xu_first_level_standalone.py",
        "visual_authority": "East_River_Plotting_Requirements_JHM_AMS_v5.md",
        "fraction_atol": FRACTION_ATOL, "mm_atol": MM_ATOL,
        "process_colors": PROCESS_COLORS,
        "versions": {"matplotlib": mpl.__version__, "numpy": np.__version__, "pandas": pd.__version__},
        "exports": [str(png_path), str(pdf_path)],
    }
    audit_path = out_dir / f"{stem}_provenance.json"
    audit_path.write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    return png_path, pdf_path, audit_path


In [ ]:
if not METRICS_PATH.is_file():
    raise FileNotFoundError(f"Set METRICS_PATH to the reviewed Xu CSV: {METRICS_PATH}")
df, audit_residuals = validate_xu_metrics(pd.read_csv(METRICS_PATH))
font_path = configure_publication_style(NIMBUS_FONT_PATH)
print(f"Verified font: {font_path}")
print(f"Validated {len(df)} rows; WY {df.water_year.tolist()}")
print("Max direct multiplicative residual:", audit_residuals["direct multiplicative closure"])
fig, ax = plot_xu(df)
saved_paths = export_xu(fig, FIGURE_OUTPUT_DIR, FIGURE_STEM, df, audit_residuals, font_path)
plt.show()
for path in saved_paths:
    print("Saved:", path)


## Styling decisions

- Nimbus Sans is checked by name and resolved with fallback disabled. Normal 18 pt labels/ticks/legend; 20 pt panel-title default (not used for this single panel). PDF fonts are embedded as TrueType.
- True grouped bars remain true bars. Gray 0.94 background, subtle dashed horizontal grid, black edges, a zero reference, and hidden top/right spines follow v5. Hatching distinguishes ablation without relying solely on color.
- The original purple/orange process palette is retained because these bars are **process terms**, not iSnobal and Noah-MP separately. East River black/blue/red remain reserved for source comparisons. The black total line is explicitly a derived comparison, not an observation.
- The original lower-left placement is retained to avoid data near the top-left, with the established white, square, gray-edged legend box. The legend now describes the total accurately as a corrected **SWE-gain** difference.
- The original 14 × 7.5 in geometry, bar order/width, total line/markers, signed axis limits, and ×100 values are preserved. Both 600 dpi PNG and vector PDF are mandatory, with a revised filename stem to protect prior figures. Inspect at the intended journal width (e.g. 6.5 in) before submission.
- No main title, extra statistics, notches, medians, or mean diamonds are added. Old embedded figure outputs have been cleared so they cannot be mistaken for revised results.
